In [1]:
import nltk
import ssl

# 处理 SSL 证书
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl.create_default_context=_create_unverified_https_context

nltk.data.path.append("../model/nltk_data")

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key=os.getenv("DEEPSEEK_API_KEY")

In [3]:
CUSTOM_CACHE = r'F:\Teewon\Milvue\models'

os.environ['HF_HOME'] = CUSTOM_CACHE
os.environ['HF_HUB_CACHE'] = os.path.join(CUSTOM_CACHE, 'hub')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(CUSTOM_CACHE, 'transformers')

os.environ['TORCH_HOME'] = CUSTOM_CACHE

In [4]:
from pymilvus import MilvusClient

mc = MilvusClient(uri="http://localhost:19530")

In [5]:
from llama_index.core import Settings,SimpleDirectoryReader,VectorStoreIndex
from llama_index.llms.deepseek import DeepSeek
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import torch

# 1. 配置 DeepSeek LLM
Settings.llm=DeepSeek(
    model="deepseek-v4-flash",
    apikey=api_key,
    temperature=0.1,
)

# 2. 配置 HuggingFace 嵌入模型
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print(DEVICE)
Settings.embed_model=HuggingFaceEmbedding(
    model_name="BAAI/bge-large-en-v1.5",
    device=DEVICE,
)

# 3. 其他设置
Settings.chunk_size=512
Settings.chunk_overlap=round(Settings.chunk_size*0.1,0)

F:\Teewon\Milvue\.venv\Lib\site-packages\transformers\utils\hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


cuda


## 准备文档数据

我们使用几篇公开的维基百科城市介绍文章作为示例文档（文本格式）。

这里直接从网络下载并保存为本地文件，再通过 `SimpleDirectoryReader` 加载。

In [11]:
import requests
from pathlib import Path

wiki_titles = ["Toronto", "Seattle", "San Francisco", "Chicago", "Boston", "Washington, D.C.", "Cambridge, Massachusetts", "Houston"]

# 创建文档目录
docs_dir=Path("./docs")
docs_dir.mkdir(exist_ok=True)

# 循环下载每个城市的维基百科摘要
headers={
    "User-Agent":"MyRAGBot/1.0 (https://myapp.com; contact@myapp.com)"
    # 维基百科要求提供 User-Agent
}
for title in wiki_titles:
    resp=requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={
            "action": "query",
            "format": "json",
            "titles": title,
            "prop": "extracts",
            # 'exintro': True,      # 取消注释可只获取引言部分
            "explaintext": True,    # 返回纯文本，不含 HTML
        },
        headers=headers,
        timeout=10,
    ).json()

    # 解析 API 返回结果
    pages=resp.get('query',{}).get('pages',[])
    if not pages:
        print(f"{title} has no pages")
        continue

    page=next(iter(pages.values()))
    wiki_text=page.get('extract','')
    if not wiki_text:
        print(f"{title} has no text")
        continue

    # 保存为 .txt 文件
    file_path=docs_dir/f"{title}.docx"
    with open(file_path,'w',encoding="utf-8") as fp:
        fp.write(wiki_text)
    print(f"下载 {title} 成功，文件大小: {len(wiki_text)} 字符")
print("\n所有文档下载完成！")

下载 Toronto 成功，文件大小: 85056 字符
下载 Seattle 成功，文件大小: 78334 字符
下载 San Francisco 成功，文件大小: 94357 字符
下载 Chicago 成功，文件大小: 90055 字符
下载 Boston 成功，文件大小: 67789 字符
下载 Washington, D.C. 成功，文件大小: 82329 字符
下载 Cambridge, Massachusetts 成功，文件大小: 55857 字符
下载 Houston 成功，文件大小: 82134 字符

所有文档下载完成！


In [12]:
# 加载文档并构建索引
documents=SimpleDirectoryReader("./docs").load_data()
print(f"共加载 {len(documents)} 个文档片段")

# 构建向量索引
index=VectorStoreIndex.from_documents(documents) # 自动使用 Settings.embed_model
print("索引构建完成")

共加载 8 个文档片段
索引构建完成


In [14]:
# 获取嵌入维度(bge-large-en-v1.5 = 1024)
# 新版 llama_index 的 HuggingFaceEmbedding 没有 .embedding_dim 属性,
# 直接编码一个句子来获取实际维度。
test_vec = Settings.embed_model.get_text_embedding("hello")
EMBEDDING_DIM = len(test_vec)
print(f"当前嵌入模型维度: {EMBEDDING_DIM}")

当前嵌入模型维度: 1024


## 持久化索引到 Milvus 并执行多文档问答

参考 milvus-io/bootcamp 的 `multi_doc_qa_llamaindex` 教程:

1. 用 `MilvusVectorStore` 把向量索引写入 Milvus(对应原教程的 `StorageContext` + `MilvusVectorStore`)
2. 用 `as_query_engine` 做单文档问答
3. 用 `DecomposeQueryTransform` + `TransformQueryEngine` 做跨文档对比问答(自动把复杂问题拆成子问题)

In [15]:
# 把索引写入 Milvus(参考原教程的 StorageContext + MilvusVectorStore)
from llama_index.vector_stores.milvus import MilvusVectorStore
from llama_index.core import StorageContext

vector_store = MilvusVectorStore(
    uri="http://localhost:19530",      # 与前面 mc 相同的 Milvus 服务
    collection_name="llamaindex_docs",
    dim=EMBEDDING_DIM,                 # bge-large-en-v1.5 = 1024
    overwrite=True,                    # 每次重建集合,便于重复运行
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# 重新构建索引并写入 Milvus(内部用 Settings.embed_model 编码,约 3-4 分钟)
index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)
print("索引已写入 Milvus 集合: llamaindex_docs")

索引已写入 Milvus 集合: llamaindex_docs


In [16]:
# 基础查询引擎:检索 top-k 片段,交给 DeepSeek 生成回答
query_engine = index.as_query_engine(similarity_top_k=3)
    # 子问题在合并索引里取 top-3,
    # Seattle/Houston 的片段总是排前面
    # Toronto 的机场片段根本没被检索到。

# 测试单个问题
response = query_engine.query("What are the main airports in Seattle?")
print(response)

The main airports in the Seattle area are Seattle-Tacoma International Airport (Sea-Tac), Boeing Field, and Paine Field.


### 跨文档对比问答(每城市独立引擎 + 汇总)

原教程用 `DecomposeQueryTransform` 把"对比多个文档"的复杂问题拆成若干子问题,

分别检索回答后由 LLM 合并为最终答案。

In [22]:
from llama_index.core.indices.query.query_transform.base import DecomposeQueryTransform
from llama_index.core.query_engine import TransformQueryEngine
from llama_index.core import PromptTemplate

custom_prompt = PromptTemplate(
    "You are a helpful assistant. Decompose the following question into 2-3 simpler sub-questions. "
    "Each sub-question should be a single sentence that can be answered independently. "
    "Output each sub-question on a new line starting with 'New question: '.\n"
    "Question: {query_str}\n"
    "Context: {context_str}\n"
    "Sub-questions:"
)

decompose_transform = DecomposeQueryTransform(Settings.llm, verbose=True, decompose_query_prompt=custom_prompt)
query_engine_decompose = TransformQueryEngine(query_engine, decompose_transform)

In [ ]:
# 跨文档对比问题:自动分解后分别检索 Seattle / Houston / Toronto 再合并

response = query_engine_decompose.query(
    "Compare and contrast the airports in Seattle, Houston, and Toronto."
)
print(response)

In [ ]:
# 跨文档对比问题 2:体育环境
response = query_engine_decompose.query(
    "Compare and contrast the sports environment of Houston and Boston."
)
print(response)